In [1]:
! pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 4.3 MB/s eta 0:00:00


In [2]:
# ============================================================
# NOTEBOOK 2 — Hyperparameter Tuning
# Models : RandomForest, SVM
# Methods : GridSearchCV, RandomizedSearchCV, Optuna
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.datasets import load_breast_cancer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

#import optuna
#from optuna.integration import SklearnPipelineSampler

import warnings
warnings.filterwarnings("ignore")

data = load_breast_cancer()
X = data.data
y = data.target


* 1. Baseline model

In [3]:
pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

pipe_svm.fit(X_train, y_train)
pipe_svm.score(X_test, y_test)


0.9790209790209791

2. GridSearchCV sur SVM

In [4]:
grid_params = {
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", "auto"],
    "model__kernel": ["rbf", "linear"]
}

grid = GridSearchCV(pipe_svm, grid_params, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Test accuracy:", grid.score(X_test, y_test))


Best parameters: {'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'linear'}
Test accuracy: 0.986013986013986


3. RandomizedSearchCV

In [5]:
# rand_params = {
#     "model__C": np.logspace(-3, 3, 20),
#     "model__gamma": np.logspace(-4, 1, 20),
#     "model__kernel": ["rbf"]
# }

rand_params = {
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", "auto"],
    "model__kernel": ["rbf", "linear"]
}

rand = RandomizedSearchCV(
    pipe_svm, rand_params, n_iter=25, cv=5, n_jobs=-1, random_state=42)
rand.fit(X_train, y_train)

print("Best parameters:", rand.best_params_)
print("Test accuracy:", rand.score(X_test, y_test))

Best parameters: {'model__kernel': 'linear', 'model__gamma': 'scale', 'model__C': 0.1}
Test accuracy: 0.986013986013986


4. Tuning avec OPTUNA (optimisation bayésienne)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)


In [6]:
import optuna
#from optuna.integration import SklearnPipelineSampler
def objective(trial):
    C = trial.suggest_loguniform("model__C", 1e-3, 1e3)
    gamma = trial.suggest_loguniform("model__gamma", 1e-4, 1)
    kernel = trial.suggest_categorical("model__kernel", ["rbf", "linear"])

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(C=C, gamma=gamma, kernel=kernel))
    ])

    pipe.fit(X_train, y_train)
    return pipe.score(X_test, y_test)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

study.best_params


[I 2026-02-02 18:54:55,774] A new study created in memory with name: no-name-d3b8f1a0-e061-40d6-8239-0321637b34cd
[I 2026-02-02 18:54:55,793] Trial 0 finished with value: 0.7482517482517482 and parameters: {'model__C': 0.07535118981552265, 'model__gamma': 0.15348249332432626, 'model__kernel': 'rbf'}. Best is trial 0 with value: 0.7482517482517482.
[I 2026-02-02 18:54:55,816] Trial 1 finished with value: 0.7272727272727273 and parameters: {'model__C': 0.9009252527321692, 'model__gamma': 0.5741552871387073, 'model__kernel': 'rbf'}. Best is trial 0 with value: 0.7482517482517482.
[I 2026-02-02 18:54:55,828] Trial 2 finished with value: 0.951048951048951 and parameters: {'model__C': 98.6710904880118, 'model__gamma': 0.03709589315162782, 'model__kernel': 'rbf'}. Best is trial 2 with value: 0.951048951048951.
[I 2026-02-02 18:54:55,838] Trial 3 finished with value: 0.9790209790209791 and parameters: {'model__C': 0.02201013368929844, 'model__gamma': 0.10101827389449439, 'model__kernel': 'line

{'model__C': 0.5175946098091763,
 'model__gamma': 0.24597276238532062,
 'model__kernel': 'linear'}

- On définit les hyperparams directement dans trial.suggest_*.

In [7]:
import optuna

def objective(trial):

    C = trial.suggest_float("C", 1e-3, 1e3, log=True)
    gamma = trial.suggest_float("gamma", 1e-4, 1, log=True)
    kernel = trial.suggest_categorical("kernel", ["rbf", "linear"])

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(C=C, gamma=gamma, kernel=kernel))
    ])

    pipe.fit(X_train, y_train)
    return pipe.score(X_test, y_test)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

study.best_params


[I 2026-02-02 18:54:59,280] A new study created in memory with name: no-name-263b8230-5736-4783-8490-9f98d5abc2d6
[I 2026-02-02 18:54:59,340] Trial 0 finished with value: 0.9790209790209791 and parameters: {'C': 2.2320903182103478, 'gamma': 0.2653354816630554, 'kernel': 'linear'}. Best is trial 0 with value: 0.9790209790209791.
[I 2026-02-02 18:54:59,389] Trial 1 finished with value: 0.9440559440559441 and parameters: {'C': 0.2869177211671131, 'gamma': 0.003930918274649535, 'kernel': 'rbf'}. Best is trial 0 with value: 0.9790209790209791.
[I 2026-02-02 18:54:59,453] Trial 2 finished with value: 0.6293706293706294 and parameters: {'C': 0.0035509663813849406, 'gamma': 0.0001744103031425759, 'kernel': 'rbf'}. Best is trial 0 with value: 0.9790209790209791.
[I 2026-02-02 18:54:59,489] Trial 3 finished with value: 0.986013986013986 and parameters: {'C': 0.9645819453997129, 'gamma': 0.035280832830604035, 'kernel': 'linear'}. Best is trial 3 with value: 0.986013986013986.
[I 2026-02-02 18:54:

{'C': 0.9645819453997129, 'gamma': 0.035280832830604035, 'kernel': 'linear'}

### Questions : go further

1️⃣ Tune a RandomForest :
      - n_estimators
      - max_depth
      - min_samples_split

2️⃣ Compare the performances Grid vs Random vs Optuna.

3️⃣ Test Optuna with 200 trees.

4️⃣ Visualize the curve of convergence Optuna.

In [8]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

#  GridSearchCV
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, n_jobs=-1)
grid_rf.fit(X_train, y_train)

# RandomizedSearchCV
random_rf = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_grid, n_iter=10, cv=5, n_jobs=-1, random_state=42)
random_rf.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   n_jobs=-1,
                   param_distributions={'max_depth': [None, 10, 20],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 200]},
                   random_state=42)

#### Test Optuna with 200 trees

In [10]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    n_estimators = 200
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )

    return cross_val_score(clf, X_train, y_train, n_jobs=-1, cv=5).mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print(f"Meilleurs paramètres Optuna: {study.best_params}")

[I 2026-02-02 18:58:43,429] A new study created in memory with name: no-name-9737ac9d-4a22-4207-abb7-b5f096699c01
[I 2026-02-02 18:58:45,381] Trial 0 finished with value: 0.9506703146374831 and parameters: {'max_depth': 3, 'min_samples_split': 9}. Best is trial 0 with value: 0.9506703146374831.
[I 2026-02-02 18:58:48,482] Trial 1 finished with value: 0.9529958960328317 and parameters: {'max_depth': 23, 'min_samples_split': 13}. Best is trial 1 with value: 0.9529958960328317.
[I 2026-02-02 18:58:52,677] Trial 2 finished with value: 0.9460191518467853 and parameters: {'max_depth': 2, 'min_samples_split': 3}. Best is trial 1 with value: 0.9529958960328317.
[I 2026-02-02 18:58:54,538] Trial 3 finished with value: 0.9529958960328317 and parameters: {'max_depth': 4, 'min_samples_split': 13}. Best is trial 1 with value: 0.9529958960328317.
[I 2026-02-02 18:58:56,423] Trial 4 finished with value: 0.9506703146374831 and parameters: {'max_depth': 13, 'min_samples_split': 17}. Best is trial 1 wit

Meilleurs paramètres Optuna: {'max_depth': 12, 'min_samples_split': 6}


In [12]:
from optuna.visualization import plot_optimization_history
from sklearn.metrics import accuracy_score

plot_optimization_history(study).show()

results = pd.DataFrame({
    "Method": ["GridSearch", "RandomSearch", "Optuna"],
    "Score": [
        grid_rf.score(X_test, y_test),
        random_rf.score(X_test, y_test),
        accuracy_score(y_test, RandomForestClassifier(n_estimators=200, **study.best_params).fit(X_train, y_train).predict(X_test))
    ]
})
print(results)

         Method     Score
0    GridSearch  0.958042
1  RandomSearch  0.958042
2        Optuna  0.958042
